# Generative Adversarial Networks
_From a histogram to handwritten digits_

This lab builds a generative adversarial network from the ground up.

Part I writes one entirely by hand, two networks held as plain tensors, two losses written
from the formula, a gradient step coded line by line, on data simple enough that every
quantity involved can be drawn on a single figure.

Part II keeps the algorithm and changes the data to MNIST.

Part III conditions the generation, so that one can ask for a particular digit rather than
taking what comes.

The object of the lab is the *mechanism*, not a difficult generation problem. A GAN is the
first model of this course that is not trained by minimizing a loss: two networks are trained
against each other, and nothing in the training curves says whether it is working. That is
what makes it worth writing by hand once.


## Conventions used in this lab

Five conventions are used throughout and are worth stating once and for all.

**1. The discriminator returns a logit, never a probability.** Its last layer is a plain
`nn.Linear`, and the probability is $D(x) = \sigma(d(x))$ where $d$ is that raw score. The
probability is formed only for display; inside the losses everything is written with
`F.softplus`, which is the numerically stable way to write $\log \sigma$; the third section
of Part I shows what happens otherwise. This is the opposite of the choice made for the
decoders of the autoencoder lab, and for the same reason: there, the output *was* the image,
and had to live in $[0, 1]$; here the output feeds a logarithm.

**2. Two objectives, two optimizers, and one never touches the other's parameters.** There is
no single loss to minimize in this lab. Every training loop alternates a discriminator step
and a generator step, each with its own optimizer, and each step ends by clearing the
gradients it left in the other network. A GAN that "does not converge" is very often a loop in
which that bookkeeping is wrong.

**3. The generator ends where the data lives.** On the toy data of Part I the output layer is
linear, because the data is unbounded. On MNIST the images are normalized to $[-1, 1]$ and the
generator ends with a $\tanh$. The rule is that the last activation and the normalization of
the data are a single decision made twice.

**4. There is nothing to early-stop on.** The losses of a GAN measure who is winning, not how
good the samples are: a generator can improve while its loss rises, and a perfectly balanced
game gives $\mathcal{L}_D \approx 2\log 2 \approx 1.39$ whether the samples are excellent or
worthless. There is no validation loss in this lab, and no model selection. What replaces them
is figures, and, on the toy data where it is possible, an explicit coverage measurement.

**5. Everything is sent to the same `device`.** The models and *all* the tensors given to
them, latent vectors included. As elsewhere in this course, a tensor left on the CPU while the
model sits on the GPU is the most common error, and it does not show up on a machine that has
no GPU.

One consequence of that device is worth expecting: `torch.manual_seed` does not drive the same
stream of numbers on a CPU and on a GPU, so the same notebook run on two machines produces
different draws. Everything asserted in this lab is about what the figures *show*, never about a
number reproducing to the digit, and your own values will differ from the ones quoted in the
answers.


## Setting up the environment


In [ ]:
import math
import random


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.transforms import v2
from torchinfo import summary

print("torch version:", torch.__version__)


In [ ]:
from tqdm import tqdm
#from tqdm.notebook import tqdm

from scipy.stats import gaussian_kde


In [ ]:
# All the tensors and models of this lab will be sent to this device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Pinning host memory only speeds up transfers towards an accelerator: without one,
# it does nothing, and recent versions of PyTorch warn about it at every DataLoader.
PIN_MEMORY = (device.type == "cuda")


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


---
# PART I: A GAN written by hand

A generative model is a machine that produces new data. In this lab it is a single function,
the *generator* $G$, applied to noise:
$$ z \sim \mathcal{N}(0, I_d) \,, \qquad x = G_\theta(z) \,. $$
Drawing a new sample is one forward pass. The distribution of the $x$ obtained this way is
called $p_g$; it is the push-forward of the Gaussian by $G_\theta$, and training means moving
$\theta$ until $p_g$ resembles the distribution $p_{\text{data}}$ of the real data.

The difficulty is that $p_g$ has **no formula**. We can draw from it as much as we like, and we
can never evaluate it at a point. A model of this kind is called *implicit*, and it rules out
everything that would have been natural: no likelihood to maximize, no density ratio to
compute, no divergence to write down and differentiate.

The idea of [[Goodfellow et al., 2014]](https://arxiv.org/abs/1406.2661) is to *learn* the
comparison instead. A second network, the *discriminator* $D$, is trained to tell real data
from generated data; the generator is trained to make it fail. Written as a single expression,
the two networks play
$$ \min_G \max_D \; V(D, G) \;=\; \mathbb{E}_{x \sim p_{\text{data}}}\big[\log D(x)\big]
   \;+\; \mathbb{E}_{z \sim p_z}\big[\log\big(1 - D(G(z))\big)\big] \,. $$

Part I implements this on one-dimensional data, where $p_{\text{data}}$, $p_g$ and $D$ can all
be drawn on the same axis.


## The data, and what we are trying to reach


The target is a mixture of two Gaussians,
$$ p_{\text{data}} = \tfrac12\,\mathcal{N}(-1.5,\ 0.3^2) \;+\; \tfrac12\,\mathcal{N}(1.5,\ 0.3^2) \,, $$
chosen so that it is not something a single Gaussian could imitate: the generator will have to
send the noise to two separate places, which is the smallest interesting task we can give it.


In [ ]:
MODES = torch.tensor([-1.5, 1.5])
SIGMA = 0.30


def sample_real(n):
    """Draw n points from the mixture, shaped (n, 1) as the networks expect."""
    which = torch.randint(0, len(MODES), (n,))
    return (MODES[which] + SIGMA * torch.randn(n)).unsqueeze(1)


def real_density(x):
    """The true density, which we happen to know here and will use only for display."""
    x = np.asarray(x)
    return sum(0.5 * np.exp(-0.5 * ((x - m) / SIGMA) ** 2) / (SIGMA * np.sqrt(2 * np.pi))
               for m in MODES.numpy())


In [ ]:
x_real = sample_real(5000)

grid = np.linspace(-4, 4, 400)
plt.figure(figsize=(8, 3.5))
plt.hist(x_real.squeeze().numpy(), bins=80, density=True, alpha=0.45, label="5000 draws")
plt.plot(grid, real_density(grid), lw=2, label=r"$p_{\mathrm{data}}$")
plt.legend()
plt.grid(alpha=0.3)
plt.title("The distribution to imitate")
plt.show()


##### <i style="color:teal">**Question:** Suppose we drop the discriminator entirely and train $G$ to minimize $\|G(z) - x\|^2$, pairing each noise draw $z$ with a real sample $x$ drawn independently. What does the trained generator produce?</i>


**[Solution]**

<!--
It produces the mean of the data, $0$ here, for every input: a point mass exactly where the
data has none.

Pairing a $z$ with an *independent* $x$ means that, conditionally on $z$, the target is a draw
from $p_{\text{data}}$. The value minimizing $\mathbb{E}\|G(z) - x\|^2$ at fixed $z$ is
therefore $\mathbb{E}[x] = 0$, whatever $z$ is. The generator has no reason to depend on its
input at all.

The failure is not an artefact of the independent pairing: it is that a per-sample distance
compares $G(z)$ to *one* real point, whereas the thing we want to make small is a distance
between two *distributions*. This is the same averaging effect that makes an autoencoder
trained with a quadratic loss return blurry reconstructions, and the reason we are about to
train a second network rather than write a loss by hand.
-->


## Two networks, held as plain tensors


Both networks are two-hidden-layer perceptrons with $\tanh$ activations. Nothing here uses
`nn.Module`: the parameters are ordinary tensors carrying `requires_grad=True`, the forward
pass is a function, and `autograd` does not care. Writing it this way once makes it visible
that a `Module` is a convenience for holding parameters, not a mathematical object.

The generator maps $\mathbb{R} \to \mathbb{R}$, the latent dimension being one, so that the map
$z \mapsto G(z)$ can be drawn as a curve. The discriminator maps $\mathbb{R} \to \mathbb{R}$
too, and returns a logit (convention 1).


In [ ]:
LATENT_DIM = 1
HIDDEN = 64


def init_mlp(n_in, n_hidden, n_out, seed=None):
    """Six tensors: two per layer, all of them leaves of the computational graph.

    The scaling by 1/sqrt(fan_in) is the usual one; with tanh activations it keeps the
    pre-activations in the non-flat part of the curve at initialization.
    """
    if seed is not None:
        torch.manual_seed(seed)

    def layer(a, b):
        W = (torch.randn(a, b, device=device) / math.sqrt(a)).requires_grad_(True)
        c = torch.zeros(b, device=device, requires_grad=True)
        return [W, c]

    return layer(n_in, n_hidden) + layer(n_hidden, n_hidden) + layer(n_hidden, n_out)


def mlp_forward(x, params):
    """Forward pass of the perceptron described by the six tensors of `params`."""
    W1, b1, W2, b2, W3, b3 = params
    h = torch.tanh(x @ W1 + b1)
    h = torch.tanh(h @ W2 + b2)
    return h @ W3 + b3


In [ ]:
g_params = init_mlp(LATENT_DIM, HIDDEN, 1, seed=SEED)
d_params = init_mlp(1, HIDDEN, 1, seed=SEED + 1)

print(f"generator     : {sum(p.numel() for p in g_params)} parameters")
print(f"discriminator : {sum(p.numel() for p in d_params)} parameters")


Before any training, the generator is already a function $z \mapsto G(z)$, an arbitrary one.
The figure below shows that curve and the distribution it induces: the Gaussian noise on the
horizontal axis is pushed through the curve and lands, on the vertical axis, wherever the curve
sends it. Training will bend this curve until the induced distribution has two modes.


In [ ]:
def plot_transport(generate, decode, ax=None, title=None):
    """The generator as a curve, and the distribution that curve induces.

    `generate(n)` returns n samples, `decode(z)` applies the generator to a given batch
    of latent vectors. Both are passed as functions so that this figure works unchanged
    for the hand-written generator and for the `nn.Module` one.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 3.5))

    z_grid = torch.linspace(-3, 3, 400, device=device).view(-1, 1)
    with torch.no_grad():
        curve = decode(z_grid).cpu().squeeze()
        fake = generate(4000).cpu().squeeze()

    ax.plot(z_grid.cpu().squeeze(), curve, lw=2, color="C1", label=r"$z \mapsto G(z)$")
    ax.set_xlabel("latent $z$")
    ax.set_ylabel("data $x$")
    ax.grid(alpha=0.3)

    twin = ax.twiny()
    twin.hist(fake.numpy(), bins=60, density=True, orientation="horizontal",
              alpha=0.35, color="C1")
    twin.plot(real_density(np.linspace(-4, 4, 300)), np.linspace(-4, 4, 300),
              color="C0", lw=2)
    twin.set_xticks([])
    twin.set_xlim(0, 2.2)
    ax.set_ylim(-4, 4)
    ax.legend(loc="upper left")
    ax.set_title(title or "The generator as a transport of the noise")
    return ax


plot_transport(lambda n: mlp_forward(torch.randn(n, LATENT_DIM, device=device), g_params),
               lambda z: mlp_forward(z, g_params),
               title="Before training: an arbitrary curve, and the distribution it induces")
plt.show()


## The two objectives, written from the formula


The discriminator maximizes $V$, which we write as a minimization of
$$ \mathcal{L}_D \;=\; -\,\mathbb{E}_{p_{\text{data}}}\big[\log D(x)\big]
   \;-\; \mathbb{E}_{p_z}\big[\log\big(1 - D(G(z))\big)\big] \,. $$
This is exactly a binary cross-entropy: real data has label $1$, generated data has label $0$.
A GAN discriminator is an ordinary classifier; the only unusual thing about it is that one of
its two classes is being rewritten at every step.

Translating that formula into code is where the first trap of this lab sits. The obvious
version applies a sigmoid and then a logarithm.


In [ ]:
def loss_d_naive(d_real, d_fake):
    """The formula, transcribed literally. Do not use."""
    return -(torch.log(torch.sigmoid(d_real)).mean()
             + torch.log(1 - torch.sigmoid(d_fake)).mean())


# A discriminator that has become confident produces logits of this size routinely
d_real_confident = torch.tensor([40.0, 35.0, 42.0])
d_fake_confident = torch.tensor([-40.0, -35.0, 42.0])

print("naive :", loss_d_naive(d_real_confident, d_fake_confident).item())


##### <i style="color:teal">**Question:** Where does that value come from, and what exactly overflowed? Write the loss so that it stays finite for logits of any size.</i>


**[Solution]**

<!--
`torch.sigmoid(42.)` rounds to exactly `1.0` in float32: the true value differs from one by
about $6 \times 10^{-19}$, far below the $10^{-7}$ resolution of the format near $1$. The
second term then computes `torch.log(1. - 1.) = -inf`, and the mean of the batch is infinite.
The first term is fine here; it would have gone the same way with a confidently *rejected*
real sample.

The fix is never to form the probability. Writing $\sigma$ for the sigmoid,
$$ -\log \sigma(d) \;=\; \log\big(1 + \mathrm{e}^{-d}\big) \;=\; \mathrm{softplus}(-d) \,,
\qquad -\log\big(1 - \sigma(d)\big) \;=\; \mathrm{softplus}(d) \,, $$
and `F.softplus` is implemented so that it returns its argument for large positive inputs
instead of computing an exponential that overflows. The loss becomes

    loss_d = F.softplus(-d_real).mean() + F.softplus(d_fake).mean()

which is the same function mathematically and finite everywhere. This is exactly what
`binary_cross_entropy_with_logits` does internally, and the reason convention 1 keeps the
discriminator's output raw.
-->


In [ ]:
def loss_discriminator(d_real, d_fake):
    """-log D(x) - log(1 - D(G(z))), in the stable form."""
    return F.softplus(-d_real).mean() + F.softplus(d_fake).mean()


print("stable:", loss_discriminator(d_real_confident, d_fake_confident).item())


For the generator, the formula in the minimax problem asks to minimize
$\mathbb{E}\big[\log(1 - D(G(z)))\big]$. In practice one minimizes
$$ \mathcal{L}_G \;=\; -\,\mathbb{E}_{p_z}\big[\log D(G(z))\big]
   \;=\; \mathbb{E}_{p_z}\big[\mathrm{softplus}\big({-}d(G(z))\big)\big] $$
instead, the *non-saturating* form, already recommended in the original article. The two have
the same optimum and very different gradients; the section on saturation measures the difference
asserting it. Until then we use the non-saturating version.


In [ ]:
def loss_generator(d_fake, saturating=False):
    """Non-saturating by default: -log D(G(z)). See the section on saturation for the other."""
    if saturating:
        return -F.softplus(d_fake).mean()      # = mean of log(1 - D(G(z)))
    return F.softplus(-d_fake).mean()          # = mean of -log D(G(z))


## One gradient step, written by hand


There is no optimizer in this section either. A step of plain gradient descent is

    p <- p - lr * dL/dp

applied to each tensor, followed by clearing the gradients so that the next `backward` does not
add to them. Both operations happen inside `torch.no_grad()`: updating a parameter is not part
of the computation being differentiated.


In [ ]:
def sgd_step(params, lr):
    """One step of gradient descent, and the gradients cleared behind it."""
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad
            p.grad = None


def zero_grad(params):
    """Drop whatever gradients are sitting in these tensors."""
    for p in params:
        p.grad = None


def grad_norm(params):
    """Euclidean norm of the full gradient, all tensors stacked.

    This is the instrument that will say which of the two networks is still learning.
    It accepts a list of tensors or a `model.parameters()` generator, so the same
    function serves the hand-written GAN and the `nn.Module` one.
    """
    return math.sqrt(sum(float((p.grad ** 2).sum()) for p in params if p.grad is not None))


The training loop alternates one discriminator step and one generator step. Read it against the
two losses above; the only things in it that are not a transcription of the formulas are the
three lines about gradient bookkeeping, and those are the ones worth looking at twice.


In [ ]:
def train_manual(g_params, d_params, n_steps=3000, batch_size=256, lr=0.05,
                 saturating=False, record_every=25, snapshot_steps=()):
    """Train the hand-written GAN, and record enough to look at it afterwards.

    `snapshot_steps` stores a copy of both networks at the given steps; the figures of
    the next section are drawn from those copies.
    """
    history = {k: [] for k in
               ["step", "loss_d", "loss_g", "grad_d", "grad_g", "d_real", "d_fake"]}
    snapshots = {}

    for step in range(n_steps):

        # ---- discriminator step ------------------------------------------------
        x_real = sample_real(batch_size).to(device)
        z = torch.randn(batch_size, LATENT_DIM, device=device)
        x_fake = mlp_forward(z, g_params).detach()   # the generator is frozen here

        d_real = mlp_forward(x_real, d_params)
        d_fake = mlp_forward(x_fake, d_params)
        loss_d = loss_discriminator(d_real, d_fake)

        loss_d.backward()
        grad_d = grad_norm(d_params)
        sgd_step(d_params, lr)                       # updates D, clears D's gradients

        # ---- generator step ----------------------------------------------------
        z = torch.randn(batch_size, LATENT_DIM, device=device)
        d_fake_g = mlp_forward(mlp_forward(z, g_params), d_params)   # no detach here
        loss_g = loss_generator(d_fake_g, saturating)

        loss_g.backward()
        grad_g = grad_norm(g_params)
        sgd_step(g_params, lr)                       # updates G, clears G's gradients
        zero_grad(d_params)                          # this backward also filled D's

        # ---- instruments -------------------------------------------------------
        if step % record_every == 0:
            history["step"].append(step)
            history["loss_d"].append(loss_d.item())
            history["loss_g"].append(loss_g.item())
            history["grad_d"].append(grad_d)
            history["grad_g"].append(grad_g)
            history["d_real"].append(torch.sigmoid(d_real).mean().item())
            history["d_fake"].append(torch.sigmoid(d_fake).mean().item())

        if step in snapshot_steps:
            snapshots[step] = ([p.detach().clone() for p in g_params],
                               [p.detach().clone() for p in d_params])

    return history, snapshots


##### <i style="color:teal">**Question:** The fake batch is detached at the discriminator step, and not at the generator step. For each of the two, say what would happen if the `detach()` were removed, or added.</i>


**[Solution]**

<!--
**Removing the `detach()` at the discriminator step changes nothing to the result, and costs
time.** Without it, `loss_d.backward()` also propagates through the generator and fills
`g_params[i].grad`. Those values are never used: the next thing that touches the generator is
its own `backward`, and by then the gradients have been cleared, either by
`sgd_step(g_params, lr)` at the end of the previous iteration, or by `opt_g.zero_grad()` in the `nn.Module` version.
So the only effect is a backward pass through a network whose parameters are not being
updated. It is a matter of compute, not of correctness, *provided* the gradients really are
cleared, which is the assumption the last two lines of the loop are there to guarantee.

**Adding a `detach()` at the generator step breaks the training completely.** The generator's
loss would then be a function of a constant: `x_fake` would carry no history, `loss_g.backward()`
would leave `g_params[i].grad` at `None`, and `sgd_step` would fail on a `None` gradient, or,
worse, in a version that skips missing gradients, would silently leave the generator at its
initial value forever. The whole point is that the gradient of the generator's loss travels
*through* the discriminator and back into $G$: the discriminator is the differentiable loss
function the generator is trained against.

Writing the `detach()` explicitly at the discriminator step is therefore not required, but it
states the intent (this batch is data, not something we are optimizing) and it is the
convention in every implementation you will read.
-->


## Training, and three ways of watching it


Three thousand steps take a few seconds. Snapshots of both networks are kept at four moments so
that the training can be replayed afterwards.


In [ ]:
SNAPSHOTS = (0, 100, 500, 2999)

g_params = init_mlp(LATENT_DIM, HIDDEN, 1, seed=SEED)
d_params = init_mlp(1, HIDDEN, 1, seed=SEED + 1)

history, snapshots = train_manual(g_params, d_params, n_steps=3000, lr=0.05,
                                  snapshot_steps=SNAPSHOTS)

print(f"final: loss_d = {history['loss_d'][-1]:.3f}, loss_g = {history['loss_g'][-1]:.3f}, "
      f"D(real) = {history['d_real'][-1]:.2f}, D(fake) = {history['d_fake'][-1]:.2f}")


### First instrument: the two distributions and the boundary between them

Each panel shows the target density, the histogram of four thousand generated points, and the
curve $x \mapsto D(x)$ on the right-hand axis. Reading the red curve is the point of the
figure: wherever it sits above $1/2$ the discriminator believes the data is real, and the
generator is being pushed towards those regions.

Two things are worth noticing on the last panel. $D$ has flattened to $1/2$ almost everywhere,
which is the equilibrium the third instrument will quantify. And $p_g$ is *spikier* than
$p_{\text{data}}$: the generator piles its mass into two narrow peaks instead of reproducing the
width of the Gaussians. The transport map drawn at the end of Part I explains why.


In [ ]:
def plot_state(g_params, d_params, ax, title=""):
    """The two densities, and the discriminator's opinion over the whole axis."""
    grid = torch.linspace(-4, 4, 400, device=device).view(-1, 1)
    with torch.no_grad():
        fake = mlp_forward(torch.randn(4000, LATENT_DIM, device=device), g_params)
        d_curve = torch.sigmoid(mlp_forward(grid, d_params))
    x = grid.cpu().squeeze().numpy()

    ax.plot(x, real_density(x), color="C0", lw=2, label=r"$p_{\mathrm{data}}$")
    ax.hist(fake.cpu().squeeze().numpy(), bins=60, range=(-4, 4), density=True,
            alpha=0.45, color="C1", label=r"$p_g$")
    ax.set_ylim(0, 1.4)
    ax.set_title(title)
    ax.grid(alpha=0.3)

    twin = ax.twinx()
    twin.plot(x, d_curve.cpu().squeeze().numpy(), color="crimson", lw=1.5, ls="--")
    twin.axhline(0.5, color="crimson", lw=0.8, alpha=0.4)
    twin.set_ylim(0, 1)
    twin.set_ylabel("$D(x)$", color="crimson")
    return twin


fig, axes = plt.subplots(1, 4, figsize=(20, 3.6))
for ax, step in zip(axes, SNAPSHOTS):
    g_snap, d_snap = snapshots[step]
    plot_state(g_snap, d_snap, ax, title=f"step {step}")
axes[0].legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()


### Second instrument: the push each generated sample receives

The generator's loss is a function of the generated points, so each of them carries a gradient
$\partial \mathcal{L}_G / \partial x$. Minimizing means moving against it, so the quantity
$-\partial \mathcal{L}_G / \partial x$ is literally the direction and the strength with which
the discriminator asks that sample to move. In one dimension it is a signed number, and the
whole field fits under the axis.

This is the figure usually drawn by hand in the lecture. Here it is computed.


In [ ]:
def sample_push(g_params, d_params, n=400, saturating=False):
    """-dL_G/dx for each generated point: where the discriminator is pushing it.

    The loss averages over the batch, so the gradient carried by one sample is 1/n of
    the total; multiplying by n undoes that, and makes the arrows comparable between
    two figures drawn with different batch sizes.
    """
    z = torch.randn(n, LATENT_DIM, device=device)
    with torch.no_grad():
        x_fake = mlp_forward(z, g_params)
    x_fake.requires_grad_(True)

    loss = loss_generator(mlp_forward(x_fake, d_params), saturating)
    push, = torch.autograd.grad(loss, x_fake)
    return x_fake.detach().cpu().squeeze(), (-push * n).cpu().squeeze()


# One scale for the four panels, so that the arrows can be compared from one to the
# next: the longest arrow of the whole figure is drawn 1.2 units of x long.
fields = {step: sample_push(*snapshots[step]) for step in SNAPSHOTS}
longest = max(float(push.abs().max()) for _, push in fields.values())
scale = longest / 1.2

fig, axes = plt.subplots(1, 4, figsize=(20, 2.2), sharey=True)
for ax, step in zip(axes, SNAPSHOTS):
    x, push = fields[step]
    ax.quiver(x.numpy(), np.zeros_like(x.numpy()),
              (push / scale).numpy(), np.zeros_like(push.numpy()),
              angles="xy", scale_units="xy", scale=1, width=0.004, alpha=0.6, color="C1")
    for m in MODES.numpy():
        ax.axvline(m, color="C0", lw=1.5, alpha=0.6)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-0.5, 0.5)
    ax.set_yticks([])
    ax.set_title(f"step {step}")
    ax.grid(alpha=0.3, axis="x")
axes[0].set_ylabel("push")
plt.suptitle(r"$-\partial \mathcal{L}_G / \partial x$, sample by sample "
             "(blue lines: the two modes of the data)", y=1.06)
plt.tight_layout()
plt.show()


### Third instrument: the two gradient norms

The losses say who is winning. The gradient norms say who is still learning, which is not the
same question and is the more useful one.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 3.8))

axes[0].plot(history["step"], history["loss_d"], label=r"$\mathcal{L}_D$")
axes[0].plot(history["step"], history["loss_g"], label=r"$\mathcal{L}_G$")
axes[0].axhline(2 * np.log(2), color="gray", ls=":", lw=1)
axes[0].axhline(np.log(2), color="gray", ls=":", lw=1)
axes[0].set_title("losses (dotted: the equilibrium values)")

axes[1].semilogy(history["step"], history["grad_d"], label=r"$\|\nabla_{\theta_D}\mathcal{L}_D\|$")
axes[1].semilogy(history["step"], history["grad_g"], label=r"$\|\nabla_{\theta_G}\mathcal{L}_G\|$")
axes[1].set_title("gradient norms")

axes[2].plot(history["step"], history["d_real"], label=r"$D(x)$, real")
axes[2].plot(history["step"], history["d_fake"], label=r"$D(G(z))$, fake")
axes[2].axhline(0.5, color="gray", ls=":", lw=1)
axes[2].set_ylim(0, 1)
axes[2].set_title("what the discriminator outputs")

for ax in axes:
    ax.set_xlabel("step")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


##### <i style="color:teal">**Question:** At the perfect equilibrium the discriminator cannot do better than chance, $D \equiv 1/2$. What are $\mathcal{L}_D$ and $\mathcal{L}_G$ worth there? Is a run whose losses sit exactly at those values a run that succeeded?</i>


**[Solution]**

<!--
With $D \equiv 1/2$,
$$ \mathcal{L}_D = -\log\tfrac12 - \log\tfrac12 = 2\log 2 \approx 1.386 \,, \qquad
   \mathcal{L}_G = -\log\tfrac12 = \log 2 \approx 0.693 \,. $$
These are the dotted lines on the first panel, and the run does end up near them.

But they say nothing about the samples. $D \equiv 1/2$ is what an *undertrained* discriminator
outputs as well as what a perfectly fooled one outputs, and the two are indistinguishable from
the loss alone: a discriminator that has learned nothing scores everything at one half whatever
the generator produces. The same numbers would appear for a generator that has collapsed onto
one of the two modes, if the discriminator has not yet found that out.

This is convention 4 in concrete form. The losses of a GAN are a relative measure of who is
ahead, and there is no monotone relationship between them and sample quality. What tells us
this run worked is the first figure, not the third one.
-->


## What the discriminator is really estimating


At a fixed generator, the discriminator's problem has a closed-form solution. Maximizing
$$ \int p_{\text{data}}(x) \log D(x) \,\mathrm{d}x + \int p_g(x) \log\big(1 - D(x)\big)\,\mathrm{d}x $$
pointwise in $D(x)$ gives
$$ D^\star(x) \;=\; \frac{p_{\text{data}}(x)}{p_{\text{data}}(x) + p_g(x)} \,. $$
The discriminator is not learning a boundary in the usual sense: it is estimating a **density
ratio**, and it is the only part of the system that ever holds information about $p_g$.

We can check this here, because both densities are available: $p_{\text{data}}$ analytically,
and $p_g$ by kernel density estimation on a large sample.

The check is only meaningful where there *is* data. Beyond the reach of the samples both
densities are numerically zero, $D^\star$ becomes the quotient of two numerical zeros, and the
curve one would draw there is the noise of the kernel estimate rather than an optimum the network
failed to match. The figure below is therefore restricted to the window the samples cover, from
their half-percentile to their 99.5th. Keep that restriction in mind: the
next question is about exactly this, the fact that a quantity built from a density ratio has
nothing to say outside the support.


In [ ]:
with torch.no_grad():
    fake = mlp_forward(torch.randn(20000, LATENT_DIM, device=device), g_params)
kde = gaussian_kde(fake.cpu().squeeze().numpy())

x = np.linspace(-4, 4, 400)
p_data, p_g = real_density(x), kde(x)
d_star = p_data / (p_data + p_g + 1e-12)

# A ratio of densities says nothing where there is no density. Beyond the reach of the
# samples both estimates are numerically zero, their quotient is the noise of the kernel
# estimate, and drawing it would invite a comparison that means nothing. D* is therefore
# shown only on the window the samples actually cover -- and note that this window keeps
# the valley between the two modes, where both densities are small but not empty, and
# where the interesting part of the comparison happens to be.
support = np.concatenate([sample_real(20000).squeeze().numpy(),
                          fake.cpu().squeeze().numpy()])
low, high = np.percentile(support, [0.5, 99.5])
covered = (x >= low) & (x <= high)

with torch.no_grad():
    d_learned = torch.sigmoid(
        mlp_forward(torch.tensor(x, dtype=torch.float32, device=device).view(-1, 1), d_params)
    ).cpu().squeeze().numpy()

plt.figure(figsize=(8, 4))
plt.plot(x[covered], d_star[covered], lw=2,
         label=r"$D^\star = p_{\mathrm{data}} / (p_{\mathrm{data}} + p_g)$")
plt.plot(x, d_learned, lw=2, ls="--", label="the learned $D$")
plt.axhline(0.5, color="gray", ls=":", lw=1)
plt.ylim(0, 1)
plt.xlabel("$x$")
plt.legend()
plt.grid(alpha=0.3)
plt.title("The discriminator against its closed-form optimum")
plt.show()


##### <i style="color:teal">**Question:** Substituting $D^\star$ back into $V$ gives $V(D^\star, G) = 2\,\mathrm{JS}(p_{\mathrm{data}} \,\|\, p_g) - \log 4$, where $\mathrm{JS}$ is the Jensen-Shannon divergence. What does that identity predict when the supports of the two distributions do not overlap at all, for instance at the very beginning of training on image data?</i>


**[Solution]**

<!--
The Jensen-Shannon divergence between two distributions with disjoint supports is $\log 2$,
*whatever* the two distributions are and however far apart they are. So $V(D^\star, G)$ is a
constant on that whole region, and its gradient with respect to the generator's parameters is
zero: the objective gives no indication of which way to move.

It is the perfect-discriminator case, and it is not a rare one. Two distributions supported on
low-dimensional manifolds of a high-dimensional space are almost surely disjoint, which is the
situation of any image GAN at initialization. With a discriminator close to optimal, an exactly
saturated generator gradient follows.

Two things prevent the disaster in practice, and they are different in kind. The non-saturating
loss of the next section keeps a usable gradient even when $D$ is winning; it does not restore
the Jensen-Shannon geometry, it replaces the objective by one that is better conditioned. And
the discriminator is never actually optimal, because it is given a handful of gradient steps
between two generator updates rather than trained to convergence. The line
$$ \text{one D step, one G step} $$
in the loop above is doing more work than it looks.

Changing the divergence itself is the other route, and the one Wasserstein GANs take
[[Arjovsky, Chintala & Bottou, 2017]](https://arxiv.org/abs/1701.07875): the earth-mover
distance between two disjoint distributions still knows how far apart they are.
-->


## Saturating or not: the choice of the generator's loss


The minimax formulation asks the generator to minimize $\log(1 - D(G(z)))$. Writing $d$ for the
logit, so that $D = \sigma(d)$, the two candidate losses and their derivatives are
$$ \mathcal{L}_G^{\text{sat}} = \log\big(1 - \sigma(d)\big) = -\,\mathrm{softplus}(d)\,,
   \qquad \frac{\partial \mathcal{L}_G^{\text{sat}}}{\partial d} = -D \,, $$
$$ \mathcal{L}_G^{\text{ns}} = -\log \sigma(d) = \mathrm{softplus}(-d)\,,
   \qquad \frac{\partial \mathcal{L}_G^{\text{ns}}}{\partial d} = -(1 - D) \,. $$

The two magnitudes, $D$ and $1 - D$, behave in opposite ways exactly where it matters. A
generator that is losing produces samples the discriminator rejects, $D(G(z)) \to 0$: the first
gradient vanishes, the second tends to $1$. The saturating loss is weakest precisely when the
generator most needs to move.


##### <i style="color:teal">**Todo:** Verify the two derivatives with `autograd`, and draw them both as a function of $D(G(z))$.</i>


In [ ]:
### TO BE COMPLETED ###

def generator_grad(d_fake, saturating):
    """Return |dL_G/dd| computed by autograd, for a batch of logits."""
    ...


In [ ]:
# %load solutions/toy/grad_saturating.py


The prediction is easy to test: train the same GAN again, changing only the generator's loss.


In [ ]:
g_sat = init_mlp(LATENT_DIM, HIDDEN, 1, seed=SEED)
d_sat = init_mlp(1, HIDDEN, 1, seed=SEED + 1)

history_sat, snapshots_sat = train_manual(g_sat, d_sat, n_steps=3000, lr=0.05,
                                          saturating=True, snapshot_steps=SNAPSHOTS)

fig, axes = plt.subplots(1, 3, figsize=(18, 3.8))
plot_state(g_params, d_params, axes[0], title="non-saturating, step 3000")
plot_state(g_sat, d_sat, axes[1], title="saturating, step 3000")

axes[2].semilogy(history["step"], history["grad_g"], label="non-saturating")
axes[2].semilogy(history_sat["step"], history_sat["grad_g"], label="saturating")
axes[2].set_xlabel("step")
axes[2].set_title(r"$\|\nabla_{\theta_G}\mathcal{L}_G\|$")
axes[2].legend()
axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()


##### <i style="color:teal">**Question:** The saturating run does not necessarily produce worse samples on this problem. Does that contradict the argument above?</i>


**[Solution]**

<!--
No, and the reason is that the argument is about a regime this problem barely enters.

The generator gradient is throttled by a factor $D/(1-D)$ relative to the non-saturating one.
That factor is close to $1$ while the discriminator is unsure, and only becomes punishing when
$D(G(z))$ approaches zero. On a one-dimensional mixture, with a discriminator given a single
small gradient step per generator step, the two distributions overlap almost from the start and
$D(G(z))$ never spends long near zero, so the two losses do similar work, and which run looks
better on a given seed is not meaningful.

What is meaningful is the gradient-norm panel, which shows the throttling itself, and it does
not depend on the outcome being different. The regime where the factor decides everything is
the one described in the previous section: high-dimensional data, near-disjoint supports, a
discriminator that wins early. That is where every implementation you will read uses the
non-saturating form, and it is why we use it for the rest of this lab.

Stating it the other way round: this section shows *why* the choice is made, on a problem too
easy to show *that* it matters. Believing an argument only when a toy example happens to
confirm it is a good way to learn the wrong lesson.
-->


## The same GAN, with `nn.Module` and `optim`


Everything above was written with six tensors per network and an update coded by hand. That was
the point; it is not how anyone writes a GAN twice. Now rebuild the same model with the tools
of the library, and check that nothing changed.

Three things are handed over, and it is worth naming them before using them:
`nn.Module` collects the parameters, so `model.parameters()` replaces the list we carried
around; `optim.Optimizer` owns the update rule, so `opt.step()` replaces `p -= lr * p.grad`;
and `opt.zero_grad()` replaces clearing `p.grad` by hand. Nothing else moves. In particular the
alternation, the `detach()` and the cross-clearing of gradients remain entirely your
responsibility: the library has no notion of two networks playing against each other.


##### <i style="color:teal">**Todo:** Write the `Generator` and `Discriminator` classes. Keep the architecture of the hand-written version, two hidden layers of 64 units, $\tanh$ activations, a linear output, and give them two extra arguments: `data_dim`, because the same classes are reused on two-dimensional data in the next section, and `activation`, defaulting to $\tanh$, because that section will want `LeakyReLU`.</i>


In [ ]:
### TO BE COMPLETED ###

class Generator(nn.Module):

    def __init__(self, latent_dim=1, data_dim=1, hidden=64, activation=None):
        super().__init__()
        self.latent_dim = latent_dim
        activation = activation if activation is not None else nn.Tanh()
        ...

    def forward(self, z):
        ...

    def sample(self, n, device=device):
        """Draw n new points: latent noise in, data out."""
        ...


In [ ]:
# %load solutions/toy/Generator.py


In [ ]:
### TO BE COMPLETED ###

class Discriminator(nn.Module):

    def __init__(self, data_dim=1, hidden=64, activation=None):
        super().__init__()
        activation = activation if activation is not None else nn.Tanh()
        ...

    def forward(self, x):
        ...


In [ ]:
# %load solutions/toy/Discriminator.py


##### <i style="color:teal">**Todo:** Write `train_gan_toy`, the exact counterpart of `train_manual`. It must fill the same `history` dictionary, so that the two runs can be compared on the same axes.</i>


In [ ]:
### TO BE COMPLETED ###

def train_gan_toy(generator, discriminator, sample_real,
                  n_steps=3000, batch_size=256, lr=0.05, lr_g=None, lr_d=None,
                  optimizer="sgd", saturating=False, n_d_steps=1, record_every=25,
                  snapshot_steps=(), device=device):
    """Train a toy GAN. Returns (history, snapshots).

    `lr` sets both learning rates; `lr_g` and `lr_d` override it for one player.
    """
    ...


In [ ]:
# %load solutions/toy/train_gan_toy.py


Same architecture, same optimizer, same learning rate: the two runs should settle in the same
place. They will not be superposed, and for two reasons worth knowing. The random draws are
consumed in a different order. And the initializations genuinely differ: `init_mlp` draws from a
normal law scaled by $1/\sqrt{\mathrm{fan\_in}}$ with zero biases, while `nn.Linear` uses the
uniform Kaiming scheme with non-zero biases, so the two runs do not start from the same point and
their gradient norms sit at different levels throughout.

What must agree is the destination: $\mathcal{L}_D$ settling at the same value, and the same two
modes at the end. A divergence *there*, and not in the noise, means the loop is not the same
algorithm.


In [ ]:
torch.manual_seed(SEED)
generator = Generator(latent_dim=LATENT_DIM, data_dim=1, hidden=HIDDEN)
discriminator = Discriminator(data_dim=1, hidden=HIDDEN)

history_module, _ = train_gan_toy(generator, discriminator, sample_real,
                                  n_steps=3000, lr=0.05, optimizer="sgd")

fig, axes = plt.subplots(1, 3, figsize=(18, 3.8))
axes[0].plot(history["step"], history["loss_d"], label="by hand")
axes[0].plot(history_module["step"], history_module["loss_d"], ls="--", label="nn.Module")
axes[0].set_title(r"$\mathcal{L}_D$")

axes[1].semilogy(history["step"], history["grad_g"], label="by hand")
axes[1].semilogy(history_module["step"], history_module["grad_g"], ls="--", label="nn.Module")
axes[1].set_title(r"$\|\nabla_{\theta_G}\mathcal{L}_G\|$")

plot_transport(lambda n: generator.sample(n, device=device),
               lambda z: generator(z),
               ax=axes[2], title="the trained transport map")

for ax in axes[:2]:
    ax.set_xlabel("step")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


##### <i style="color:teal">**Question:** Look at the curve $z \mapsto G(z)$ on the right-hand panel. Why does it have that shape, and what would it look like if the data had three modes instead of two?</i>


**[Solution]**

<!--
It is a step: a plateau near one mode, a steep passage, a plateau near the other. Which mode
ends up on which side of the passage is decided by the initialization and means nothing; what
matters is that there is exactly one passage, and that it is steep.

That is the only shape that can push a unimodal Gaussian onto a bimodal target. The plateaus are
where the mass accumulates: a wide interval of $z$ is mapped onto a narrow interval of $x$, so
the induced density is high there. The steep part is where almost no mass lands, which is what
makes the valley between the two modes.

Look closely and the plateaus are not flat. The curve overshoots just before and just after the
passage, reaching beyond both mode centres, then drifts slowly back. And where a plateau is
flattest the density piles up, which is the answer to the observation left open when the two
densities were first drawn:
the peaks of $p_g$ are taller and narrower than those of $p_{\text{data}}$ because the generator
concentrates its mass rather than reproducing the width of the Gaussians. Matching a width is a
harder thing to ask of a transport map than matching a location.

With three modes the curve would have two steps, and the widths of the three plateaus would
follow the weights of the mixture: a mode carrying half the mass needs a plateau covering half
of the Gaussian's mass in $z$.

This is worth keeping in mind for the rest of the lab: the generator does its work through the
*derivative* of this map, and the sharper the target's separation, the steeper the passage it
has to build. A generator that fails to produce one of the modes is very often a generator
whose curve never grew the step it needed.
-->


The learning rate of $0.05$ used so far is large, and plain gradient descent tolerates it here
only because the problem is one-dimensional. Everything after this section uses Adam with
$\beta_1 = 0.5$, which is the setting recommended by
[[Radford, Metz & Chintala, ICLR 2016]](https://arxiv.org/abs/1511.06434): a GAN loss landscape
is rewritten at every step by the opponent's update, and the default momentum of $0.9$ carries
too much of a gradient that has since become wrong.


##### <i style="color:teal">**Todo:** Train the same pair with `optimizer="adam"` and `lr=1e-3`, and compare the number of steps needed.</i>


In [ ]:
### TO BE COMPLETED ###

torch.manual_seed(SEED)
generator_adam = ...


**[Solution]**

<!--
    torch.manual_seed(SEED)
    generator_adam = Generator(latent_dim=LATENT_DIM, data_dim=1, hidden=HIDDEN)
    discriminator_adam = Discriminator(data_dim=1, hidden=HIDDEN)

    history_adam, _ = train_gan_toy(generator_adam, discriminator_adam, sample_real,
                                    n_steps=1500, lr=1e-3, optimizer="adam")

    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
    plot_transport(lambda n: generator_adam.sample(n, device=device),
                   lambda z: generator_adam(z), ax=axes[0],
                   title="Adam, 1500 steps")
    axes[1].plot(history_module["step"], history_module["d_fake"], label="SGD, lr = 0.05")
    axes[1].plot(history_adam["step"], history_adam["d_fake"], label="Adam, lr = 1e-3")
    axes[1].axhline(0.5, color="gray", ls=":", lw=1)
    axes[1].set_xlabel("step"); axes[1].set_ylabel("$D(G(z))$")
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

Adam reaches the same result in roughly half the steps, with a learning rate fifty times
smaller: the per-parameter scaling is doing the work that the large step size was doing
before. The comparison is only indicative: the two optimizers do not have comparable learning
rates, and the honest statement is about the shape of the curves, not about a ratio of step
counts.
-->


## Two dimensions, eight modes


The same two classes, with `data_dim=2`, on a ring of eight Gaussians. Nothing in the algorithm
changes; what changes is that the failure we are about to provoke is impossible to miss in two
dimensions and hard to see in one.

Two settings change with the data, and neither is part of the algorithm. The latent dimension
moves from $1$ to $16$: it no longer has to be small, since we are not drawing the transport map
any more, and a generator whose latent space is smaller than the structure it must produce is
handicapped for a reason that has nothing to do with the adversarial game. And the activation
becomes `LeakyReLU(0.2)` in both networks, the usual choice in a GAN. The $\tanh$ of the
previous section was a concession to plain gradient descent with a large step size; Adam does
not need it, and the rectified networks are the ones whose behaviour you will recognize in
every implementation you read.


In [ ]:
N_MODES = 8
RADIUS = 2.0
SIGMA_RING = 0.10
LATENT_DIM_2D = 16

angles = torch.arange(N_MODES) * 2 * math.pi / N_MODES
CENTERS = torch.stack([RADIUS * torch.cos(angles), RADIUS * torch.sin(angles)], dim=1)


def sample_ring(n):
    """n points from an equal mixture of eight Gaussians placed on a circle."""
    which = torch.randint(0, N_MODES, (n,))
    return CENTERS[which] + SIGMA_RING * torch.randn(n, 2)


In [ ]:
def plot_ring(fake=None, discriminator=None, ax=None, title="", n_real=2000):
    """Real points, generated points, and the discriminator's opinion behind them."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))

    if discriminator is not None:
        side = torch.linspace(-3, 3, 200)
        xx, yy = torch.meshgrid(side, side, indexing="xy")
        grid = torch.stack([xx.flatten(), yy.flatten()], dim=1).to(device)
        with torch.no_grad():
            proba = torch.sigmoid(discriminator(grid)).view(200, 200).cpu()
        ax.imshow(proba, origin="lower", extent=(-3, 3, -3, 3), cmap="RdBu",
                  vmin=0, vmax=1, alpha=0.55)

    real = sample_ring(n_real)
    ax.scatter(real[:, 0], real[:, 1], s=4, alpha=0.25, color="C0", label="real")
    if fake is not None:
        fake = fake.detach().cpu()
        ax.scatter(fake[:, 0], fake[:, 1], s=4, alpha=0.35, color="C1", label="generated")

    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.legend(loc="upper right", markerscale=3, fontsize=8)
    return ax


##### <i style="color:teal">**Todo:** Train a GAN on `sample_ring` with the classes written above. Three thousand steps with Adam at `lr=1e-3` are enough; keep snapshots so that the eight modes can be watched appearing.</i>


In [ ]:
### TO BE COMPLETED ###

torch.manual_seed(SEED)
generator_2d = ...
discriminator_2d = ...

history_2d, snapshots_2d = ...


In [ ]:
# %load solutions/toy/train_ring.py


The background of the last figure is $D(x, y)$ over the whole plane: red where the
discriminator expects real data, blue where it expects fakes. A well-trained pair leaves it
nearly uniform: the discriminator has nothing left to say.

Look at the clusters before reading on. Most of the eight carry a comparable share of the
generated points, and two or three of them are visibly thin, a few dozen samples out of five
thousand against seven or eight hundred for their neighbours. Depending on the seed and on the
machine, `mode_coverage` will report seven or eight modes covered, because a mode that thin sits
right at the one per cent threshold the function uses.

Notice also that the generated clusters do not sit exactly on the real ones, several being
displaced by a fraction of a standard deviation, and that a thin trail of points runs around the
circle between them. That trail is the trace of a transport map that has to pass *through* the
empty space to reach the next mode, the two-dimensional version of the steep passage seen on
the transport map.

None of this is a bug in the run. It will matter in a moment: the failure of the next section is
this same imbalance, taken to its end.


### Counting what the losses do not count

In one dimension a histogram was enough to see whether both modes were covered. Here we can do
better than looking, because we know where the modes are.


##### <i style="color:teal">**Todo:** Write `mode_coverage`, which returns the number of modes that actually receive samples and the share of samples that land near some mode.</i>


In [ ]:
### TO BE COMPLETED ###

def mode_coverage(samples, centers=CENTERS, tol=3.0, sigma=SIGMA_RING, min_share=0.01):
    """Return (modes covered, share of samples near a mode, count per mode)."""
    ...


In [ ]:
# %load solutions/toy/mode_coverage.py


## Mode collapse, on purpose


A generator has no term in its objective that rewards diversity. It is asked, sample by sample,
to produce something the current discriminator accepts; producing the *same* accepted thing
every time satisfies that objective perfectly. What normally prevents it is the discriminator:
if all the generated mass piles onto one mode, that mode becomes a reliable sign of a fake, the
discriminator learns to reject it, and the generator has to move on.

So mode collapse is what happens when that correction stops working, when the discriminator
can no longer punish the generator's concentration, or when the generator cannot react to the
punishment. Both can be arranged.


##### <i style="color:teal">**Todo:** Break the training on purpose, in two different ways, and measure the result with `mode_coverage`. Two settings that work: a generator made twenty times slower than the discriminator (`lr_d=1e-3`, `lr_g=5e-5`, 3000 steps), and a discriminator given five steps per generator step and a larger one (`n_d_steps=5`, `lr_d=4e-3`, `lr_g=1e-3`, 2000 steps).</i>


In [ ]:
### TO BE COMPLETED ###

torch.manual_seed(SEED)
generator_slow = ...


In [ ]:
# %load solutions/toy/collapse.py


##### <i style="color:teal">**Question:** Compare the three panels on the two numbers `mode_coverage` returns. By how much does the share of samples landing near a mode change between the balanced run and the broken ones, and by how much does the number of modes covered change? What does that comparison say about judging a generative model by a per-sample quality measure?</i>


**[Solution]**

<!--
That a per-sample quality measure is very nearly blind to mode collapse.

The share near a mode barely moves. A few points either way around the balanced run's value,
sometimes above it and sometimes below, depending on which run and which seed. The coverage, over
those same runs, falls from seven or eight modes to two. One of the two numbers records a
catastrophe and the other does not notice it.

They are blind to different things because they measure different objects. The collapsed
generator produces its one or two modes *well*: each individual sample is close to a real data
point, so any score that asks "is this sample plausible?" returns a comfortable number, whether
it is the share near a mode here, the Inception Score, a discriminator's own confidence, or a
human glancing at a grid of images. What has been destroyed is a property of the *set* of
samples, not of any one of them: three quarters of the distribution is missing, and no
individual sample is evidence of it.

This is the reason mode coverage is measured separately here, and the reason the
Fréchet Inception Distance [[Heusel et al., NeurIPS 2017]](https://arxiv.org/abs/1706.08500)
replaced the Inception Score for image GANs: it compares the two distributions through their
first two moments in a feature space, so dropping a mode moves it. On MNIST, in Part II, we
will have no equivalent of the eight centres to count against, which is worth remembering
when we judge the digits by looking at them.

The two broken runs also say two different things about *why* it happened. With a slow
generator, the discriminator is effectively optimal at all times: the generator is descending a
landscape that is re-drawn faster than it can move, and the only stable behaviour left is to sit
on the safest point. With five discriminator steps per generator step, the same thing happens by
another route. In both cases the pathology is an imbalance, not a bad architecture, which is
why so much of GAN practice is about keeping the two players at a comparable strength.
-->


---
# PART II: MNIST

Nothing in the algorithm changes in this part. The same alternation, the same two losses, the
same bookkeeping; the toy sampler is replaced by a `DataLoader` and the two networks grow. What
changes is everything around it: the run now takes minutes rather than seconds, and there is no
`mode_coverage` to call, so we are back to judging by eye, with the warning of the section on
mode collapse in mind.


## The data, normalized to $[-1, 1]$


The autoencoder lab keeps the pixels in $[0, 1]$ and ends the decoder with a sigmoid. Here the
images are mapped to $[-1, 1]$ and the generator ends with a $\tanh$ (convention 3).

The reason is not cosmetic. The generator's output layer has to be able to produce the extremes
of the range, MNIST being mostly saturated black with saturated white strokes, and a $\tanh$
centred on zero reaches both ends symmetrically from a pre-activation near zero, which is where
an initialized layer sits. With a sigmoid the black background, at $0$, is at the flat end of
the curve, and the first epochs are spent crawling there.


In [ ]:
BATCH_SIZE = 128
LATENT_DIM = 100

transform = v2.Compose([
    v2.ToImage(),                           # PIL image -> tensor, with a channel dimension
    v2.ToDtype(torch.float32, scale=True),  # uint8 in [0, 255] -> float32 in [0, 1]
    v2.Normalize((0.5,), (0.5,)),           # [0, 1] -> [-1, 1]
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)

train_loader = DataLoader(
    train_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = 2,
    pin_memory = PIN_MEMORY,
    drop_last = True)     # a half-empty last batch destabilizes the batch normalizations

print(f"Train set: {len(train_dataset)} images")


In [ ]:
def to_display(image):
    """From the [-1, 1] of the models to the [0, 1] matplotlib expects."""
    return (image.detach().cpu().squeeze() + 1) / 2


def plot_grid(images, n_row=8, n_col=8, title=None, figsize=None):
    """A grid of images, drawn in the order given."""
    figsize = figsize or (n_col, n_row)
    fig, axes = plt.subplots(n_row, n_col, figsize=figsize)
    for ax, image in zip(np.array(axes).flat, images):
        ax.imshow(to_display(image), cmap="gray", vmin=0, vmax=1)
        ax.grid(False)
        ax.axis("off")
    if title:
        fig.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.show()


images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, range [{images.min():.1f}, {images.max():.1f}]")
plot_grid(images, n_row=2, n_col=10, title="Real images, as the discriminator sees them",
          figsize=(20, 4))


## A dense GAN


The first pair is made of perceptrons, as the first autoencoder of the autoencoder lab is. It is
not the right architecture for images, and that is the point of having it: the convolutional
version of the next section can then be compared against something.


##### <i style="color:teal">**Todo:** Write `GeneratorMLP`: a latent vector of dimension 100 in, an image $1 \times 28 \times 28$ in $[-1, 1]$ out. Three hidden layers of widths 256, 512 and 1024, `LeakyReLU(0.2)` between them.</i>


In [ ]:
### TO BE COMPLETED ###

class GeneratorMLP(nn.Module):

    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = latent_dim
        ...

    def forward(self, z):
        ...

    def sample(self, n, device=device):
        ...


In [ ]:
# %load solutions/mnist/GeneratorMLP.py


##### <i style="color:teal">**Todo:** Write `DiscriminatorMLP`: an image in, one logit out. Two hidden layers of widths 512 and 256, `LeakyReLU(0.2)`, and dropout.</i>


In [ ]:
### TO BE COMPLETED ###

class DiscriminatorMLP(nn.Module):

    def __init__(self, p_drop=0.3):
        super().__init__()
        ...

    def forward(self, x):
        ...


In [ ]:
# %load solutions/mnist/DiscriminatorMLP.py


In [ ]:
summary(GeneratorMLP().to(device), input_size=(BATCH_SIZE, LATENT_DIM))


In [ ]:
summary(DiscriminatorMLP().to(device), input_size=(BATCH_SIZE, 1, 28, 28))


##### <i style="color:teal">**Todo:** Write `train_gan_mnist`. It is `train_gan_toy` with a `DataLoader` in place of `sample_real`, Adam with $\beta_1 = 0.5$, and a *fixed* batch of latent vectors decoded at the end of every epoch.</i>


In [ ]:
### TO BE COMPLETED ###

def train_gan_mnist(generator, discriminator, loader, n_epochs=20, lr=2e-4,
                    saturating=False, device=device, n_preview=64, verbose=True):
    """Returns (history, previews), previews being one grid of images per epoch."""
    ...


In [ ]:
# %load solutions/mnist/train_gan_mnist.py


In [ ]:
torch.manual_seed(SEED)
generator_mlp = GeneratorMLP()
discriminator_mlp = DiscriminatorMLP()

history_mlp, previews_mlp = train_gan_mnist(generator_mlp, discriminator_mlp,
                                            train_loader, n_epochs=30)


In [ ]:
plot_grid(previews_mlp[-1], n_row=8, n_col=8,
          title=f"Dense GAN, after {len(previews_mlp)} epochs")

fig, axes = plt.subplots(1, 4, figsize=(20, 5.4))
shown = np.linspace(0, len(previews_mlp) - 1, 4).astype(int)
for ax, epoch in zip(axes, shown):
    grid = previews_mlp[epoch][:16]
    canvas = torch.cat([torch.cat(list(grid[4 * i:4 * i + 4]), dim=2) for i in range(4)], dim=1)
    ax.imshow(to_display(canvas), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"epoch {epoch + 1}")
    ax.grid(False); ax.axis("off")
plt.suptitle("The same sixteen latent vectors, decoded as training goes on", y=1.02)
plt.tight_layout()
plt.show()


##### <i style="color:teal">**Question:** The previews are made from a latent batch fixed once and for all, rather than redrawn at each epoch. What would be lost by redrawing it?</i>


**[Solution]**

<!--
The ability to see anything move.

With a fixed batch, each of the sixty-four positions in the grid follows the *same* latent
vector over the whole training, so the sequence of grids is a film of sixty-four images being
learned: a blob becomes a stroke, a stroke becomes a digit, and one can watch a given position
hesitate between a 3 and a 8. With a redrawn batch each grid is a fresh sample and successive
grids have nothing in common, so the only readable thing is the average quality, and the
early epochs, where everything is a blob, all look alike.

There is a second, quieter benefit: a fixed batch makes mode collapse visible as sixty-four
positions converging towards the same image, which a redrawn batch would show only as a
suspicious lack of variety.
-->


## DCGAN: the same game, convolutions this time


The dense generator has to learn from scratch that a pixel is related to its neighbours. The
convolutional architecture of
[[Radford, Metz & Chintala, ICLR 2016]](https://arxiv.org/abs/1511.06434) gives it that for
free, and the recipe that paper settled on is still the default starting point:

* the resolution is changed by strided convolutions, never by pooling or upsampling layers;
* batch normalization on every layer *except* the generator's output and the discriminator's
  input;
* `ReLU` in the generator, `LeakyReLU` in the discriminator;
* $\tanh$ on the generator's output;
* Adam, $\mathrm{lr} = 2\cdot 10^{-4}$, $\beta_1 = 0.5$.

The new layer is `nn.ConvTranspose2d`, which does the opposite of a strided convolution: with a
kernel of $4$, a stride of $2$ and a padding of $1$, it doubles the side of its input.


##### <i style="color:teal">**Todo:** Write `DCGenerator`. The latent vector is projected onto a $7 \times 7$ map with $2f$ channels, then two transposed convolutions bring it to $14 \times 14$ and $28 \times 28$, with $f = 64$.</i>


In [ ]:
### TO BE COMPLETED ###

class DCGenerator(nn.Module):

    def __init__(self, latent_dim=LATENT_DIM, features=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.features = features
        ...

    def forward(self, z):
        ...

    def sample(self, n, device=device):
        ...


In [ ]:
# %load solutions/mnist/DCGenerator.py


##### <i style="color:teal">**Todo:** Write `DCDiscriminator`, the mirror image: two strided convolutions bringing $28 \times 28$ down to $7 \times 7$, then a linear layer to a single logit.</i>


In [ ]:
### TO BE COMPLETED ###

class DCDiscriminator(nn.Module):

    def __init__(self, features=64):
        super().__init__()
        ...

    def forward(self, x):
        ...


In [ ]:
# %load solutions/mnist/DCDiscriminator.py


In [ ]:
summary(DCGenerator().to(device), input_size=(BATCH_SIZE, LATENT_DIM))


In [ ]:
torch.manual_seed(SEED)
generator_dc = DCGenerator()
discriminator_dc = DCDiscriminator()

history_dc, previews_dc = train_gan_mnist(generator_dc, discriminator_dc,
                                          train_loader, n_epochs=20)


In [ ]:
plot_grid(previews_dc[-1], n_row=8, n_col=8,
          title=f"DCGAN, after {len(previews_dc)} epochs")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (hist, name) in zip(axes, [(history_mlp, "dense"), (history_dc, "DCGAN")]):
    ax.plot(hist["loss_d"], label=r"$\mathcal{L}_D$")
    ax.plot(hist["loss_g"], label=r"$\mathcal{L}_G$")
    ax.axhline(2 * np.log(2), color="gray", ls=":", lw=1)
    ax.set_xlabel("epoch"); ax.set_title(name); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


##### <i style="color:teal">**Question:** Put the two sets of samples side by side, and the two pairs of loss curves side by side. Which comparison tells you which model is better?</i>


**[Solution]**

<!--
Only the first one, and the second comparison actively points the wrong way.

The samples are unambiguous: the convolutional generator produces strokes of even thickness and
closed loops, the dense one produces digits surrounded by isolated bright pixels and edges that
do not join, because nothing in its architecture relates a pixel to its neighbour: every one of
the 784 outputs is a separate linear unit.

Now read the curves as if you had not seen the images. The dense run looks healthy. After a noisy
start its two losses converge quietly towards the dotted equilibrium lines, $\mathcal{L}_D$
rising and $\mathcal{L}_G$ falling until both settle. The DCGAN run looks broken. Its
discriminator wins comfortably and stays there, scoring real images near $0.9$ and fakes near
$0.1$ with $\mathcal{L}_D$ far below the equilibrium, and its $\mathcal{L}_G$ *rises* steadily
over the twenty epochs rather than falling. A generator loss that climbs monotonically for a
whole training is the textbook picture of a run going wrong, and here it belongs to the model
whose digits are far better.

The two readings cannot be reconciled, because they are not measuring the same thing.
$\mathcal{L}_D$ and $\mathcal{L}_G$ are computed against *that run's own* discriminator, so
comparing them across two runs compares two different instruments. And inside a single run, the
generator's loss rises whenever its opponent is improving faster than it is, which is not the
same event as the samples getting worse. A dense generator facing a dense discriminator and a
convolutional generator facing a convolutional discriminator are two separate games, and the
scores of two separate games do not compare.

This is convention 4 at its most concrete, and it is the practical reason the GAN literature
needed the Fréchet Inception Distance: an external, fixed instrument, the same for every model.
-->


## Walking in the latent space


The generator is a continuous function of $z$, so a path in the latent space becomes a path in
image space. Decoding the segment between two latent vectors is the standard way of looking at
what the generator has built.


##### <i style="color:teal">**Todo:** Write `interpolate`, which decodes ten points evenly spaced between two latent vectors, and display the result.</i>


In [ ]:
### TO BE COMPLETED ###

def interpolate(generator, z_start, z_end, n_steps=10, device=device):
    """Decode the straight segment between two latent vectors."""
    ...


In [ ]:
# %load solutions/mnist/interpolate.py


##### <i style="color:teal">**Question:** The intermediate images are digit-like rather than superpositions of two digits. What does that prove about the model, and what does it not prove?</i>


**[Solution]**

<!--
It proves that the generator is a smooth map and that a whole region of the latent space, not
just a few isolated points, decodes to something digit-shaped. Had the model memorized a
finite set of images, the segment between two of them would have crossed a no man's land of
grey mush; the fact that it does not is genuine evidence that the generator has built a
continuous family.

It proves nothing at all about coverage. A generator collapsed onto three of the ten digits
would give exactly as smooth an interpolation between two of those three, and the figure would
look just as convincing. Smoothness is a property of $G$ as a function; mode collapse is a
property of the *image measure* $p_g$. The interpolation sees the first and is blind to the
second, which is the mode-collapse lesson again, transposed to a figure that is much more
seductive.

A remark on the interpolation itself. In dimension 100, a standard Gaussian puts almost all of
its mass on a thin shell of radius $\sqrt{100} = 10$, while the midpoint of two independent
draws has an expected norm of about $\sqrt{50} \approx 7$. The middle of the segment is
therefore in a region the generator saw very little of during training, and on richer datasets
this is visible as a loss of contrast halfway through. A spherical interpolation, which keeps
the norm constant, is the usual fix.
-->


## Has it simply copied the training set?


A generator that returned training images verbatim would fool the discriminator perfectly and
would be worthless. The cheapest check is to look, for a few generated images, at the closest
training image in the pixel distance.


##### <i style="color:teal">**Todo:** Write `nearest_neighbours` and display eight generated images above their nearest training neighbours.</i>


In [ ]:
### TO BE COMPLETED ###

def nearest_neighbours(images, dataset, n_reference=10000, device=device):
    """Return (the nearest training image for each input, the distances)."""
    ...


In [ ]:
# %load solutions/mnist/nearest_neighbours.py


##### <i style="color:teal">**Question:** The neighbours are visibly different images: same digit, different handwriting. Is that enough to conclude that the model is not memorizing?</i>


**[Solution]**

<!--
No. It rules out the crudest form of copying, and nothing beyond it.

The test is run on a handful of generated images, so it can only detect memorization if the
sample happens to land on a memorized image; a generator that reproduces one per cent of the
training set exactly would pass this figure most of the time. And the distance used is the
Euclidean distance between raw pixels, which is a poor measure of similarity for images: a
one-pixel translation of a digit is far, in that distance, from the original, so a generator
that copied training images and shifted them slightly would show comfortable distances and look
innocent.

What a serious answer requires is a comparison of *distributions of distances*: the distance
from a generated image to its nearest training neighbour should be compared to the distance
from a held-out *test* image to its nearest training neighbour. If the first is systematically
smaller, the model is closer to the training set than real data is, which is what memorization
means. That comparison is worth doing, and it is a good exercise to add on your own.
-->


## What this model does not give you


The GAN generates, and that is all it does. Three things it cannot do are worth naming, because
the VAE lab is built on the first two.

**It cannot tell you the probability of anything.** There is no $p_g(x)$ to evaluate, so the
model cannot say whether a new image is likely under it, cannot be compared to another model by
likelihood, and cannot be used for anomaly detection the way the VAE lab's model can.

**It cannot go from an image back to a latent vector.** There is no encoder. Asking "which $z$
would produce this image?" requires an optimization over $z$, image by image.

**It gives no signal that training is going well.** This has been convention 4 throughout, and
it is the reason we could count modes on the toy data and could only look at MNIST.

The VAE lab builds a model that has all three (a decoder that is also a probabilistic model,
an encoder, and a loss that means something) and pays for them with a fourth property. The
figure below is that price, shown before the explanation: on the left, digits from the DCGAN
you just trained; on the right, digits from the model of the VAE lab.


In [ ]:
N_PLATE = 16
vae_samples = torch.from_numpy(np.load("assets/vae_samples.npy"))[:N_PLATE]  # in [-1, 1]
gan_samples = generator_dc.sample(N_PLATE, device=device).detach().cpu()

fig, axes = plt.subplots(2, 1, figsize=(16, 2.6))
for ax, batch, name in zip(axes, [gan_samples, vae_samples], ["GAN (this lab)", "VAE lab"]):
    canvas = torch.cat(list(batch), dim=2)
    ax.imshow(to_display(canvas), cmap="gray", vmin=0, vmax=1)
    ax.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout()
plt.show()


##### <i style="color:teal">**Question:** Without knowing anything about the second model, describe precisely what distinguishes the two rows. Keep your answer: the VAE lab opens on it.</i>


**[Solution]**

<!--
The second row is blurry, and blurry in a particular way: the strokes have soft edges, the
contrast is lower, and several images are visibly an average of two possible digits: a shape
that is neither a 3 nor a 8 but lies between them. The first row is sharp, with clean black and
saturated white, and every image commits to being one digit; when it fails, it fails by
producing a shape that is not a digit at all, not by hedging.

That difference is the whole trade. The second model is trained to maximize a per-pixel
likelihood of the *real* images, so when several outputs are plausible for a given input it
returns their average, the same averaging effect as the question at the very start of Part I,
where minimizing a quadratic distance to an independently drawn sample gave the mean of the
data. The GAN is trained against a critic that has learned what a real stroke looks like, so an
average of two digits is exactly the kind of thing that critic rejects; the GAN is pushed onto
the data manifold rather than towards the middle of it.

And the price of sharpness is everything listed above this figure: the blurry model can tell
you the probability of an image, can encode one, and has a loss that decreases when it is
learning. Which of the two you want depends on what you are going to do with the model. And
the models that ended up dominating, diffusion models, are built on the second one's
foundations while recovering the first one's sharpness.
-->


---
# PART III: Asking for a particular digit <small style="color:orangered">(to go further)</small>

The generator of Part II takes noise and returns *a* digit. There is no way to ask it for a
seven: the only handle is $z$, and which digit a given $z$ produces was decided by the training,
not by us.

The conditional GAN [[Mirza & Osindero, 2014]](https://arxiv.org/abs/1411.1784) fixes this with
one idea, applied twice. Both networks receive the label:
$$ x = G(z, y) \,, \qquad D(x, y) \in [0, 1] \,. $$
The discriminator's question becomes "is this a real image *of the digit $y$*?", and that is
what does the work. A generator that answers every request with the same convincing eight is now
caught immediately, because the eight is presented with the label $3$ and rejected.


## Passing the label to both networks


The label is an integer between $0$ and $9$, and the two networks need it in different shapes.

In the generator it becomes a vector, through `nn.Embedding`, concatenated to the latent vector.
An embedding rather than the integer itself because the digits have no order ($7$ is not
between $6$ and $8$ in any sense the network should exploit), and rather than a one-hot vector
because an embedding *is* a one-hot vector followed by a linear layer, written efficiently.

In the discriminator it becomes an extra input channel of the image, so that the very first
convolution can compare the label with the pixels. Handing the label only to the final linear
layer would let the convolutional part ignore it entirely.


##### <i style="color:teal">**Todo:** Write `CondGenerator`: the DCGAN generator with an embedded label concatenated to the latent vector.</i>


In [ ]:
### TO BE COMPLETED ###

class CondGenerator(nn.Module):

    def __init__(self, latent_dim=LATENT_DIM, n_classes=10, embed_dim=32, features=64):
        super().__init__()
        ...

    def forward(self, z, labels):
        ...

    def sample(self, labels, device=device):
        """One image per entry of `labels`."""
        ...


In [ ]:
# %load solutions/cgan/CondGenerator.py


##### <i style="color:teal">**Todo:** Write `CondDiscriminator`: the DCGAN discriminator with the label as a second channel.</i>


In [ ]:
### TO BE COMPLETED ###

class CondDiscriminator(nn.Module):

    def __init__(self, n_classes=10, features=64):
        super().__init__()
        ...

    def forward(self, x, labels):
        ...


In [ ]:
# %load solutions/cgan/CondDiscriminator.py


##### <i style="color:teal">**Todo:** Write `train_cgan`. Three lines change with respect to `train_gan_mnist`, and they are all the same line: the label goes wherever the image goes.</i>


In [ ]:
### TO BE COMPLETED ###

def train_cgan(generator, discriminator, loader, n_epochs=20, lr=2e-4,
               device=device, verbose=True):
    """Returns (history, previews), previews being a 10x10 grid, one digit per row."""
    ...


In [ ]:
# %load solutions/cgan/train_cgan.py


In [ ]:
torch.manual_seed(SEED)
generator_c = CondGenerator()
discriminator_c = CondDiscriminator()

history_c, previews_c = train_cgan(generator_c, discriminator_c, train_loader, n_epochs=20)


In [ ]:
plot_grid(previews_c[-1], n_row=10, n_col=10, figsize=(10, 10),
          title="One digit per row, ten latent vectors per column")


##### <i style="color:teal">**Todo:** Write `sample_digits` and ask the model for ten sevens.</i>


In [ ]:
### TO BE COMPLETED ###

def sample_digits(generator, digit, n=10, device=device):
    """n images of the same digit, from n different latent vectors."""
    ...


In [ ]:
# %load solutions/cgan/sample_digits.py


##### <i style="color:teal">**Question:** The conditional model is usually easier to train than the unconditional one of Part II, and collapses less. Give the reason, and say what is *not* fixed.</i>


**[Solution]**

<!--
The reason is that the discriminator's task has become harder in a way that helps.

An unconditional discriminator only asks "is this a real image?". A generator that produces one
excellent digit and nothing else satisfies it, and is only punished indirectly, once the
discriminator notices that this particular digit is over-represented among the fakes, a slow,
statistical signal. The conditional discriminator asks "is this a real image of a $3$?", and a
generator that answers with an eight is rejected immediately, on that single example. The
pressure towards diversity becomes part of the per-sample loss instead of being an emergent
property of the game.

There is a second effect, unrelated to the game: the label carries information the generator
would otherwise have had to infer. The latent space no longer has to encode *which* digit and
*how* it is written; it only has to encode the second. That is a genuinely easier function to
learn.

What is not fixed is collapse *inside* a class. Nothing in the conditional loss requires the ten
sevens of the last figure to differ from one another, and a generator that produced the same
seven for every $z$ would be accepted by the discriminator every time. This is why
`sample_digits` prints the mean pairwise distance within the batch: the figure shows ten
different sevens, the number is what would tell you if it stopped being true. Conditioning moves
the problem from "which modes exist" to "how much variety within a mode", and the second is
harder to see.
-->


---
## Where this leads

Three directions, in the order they matter for the rest of the course.

**Changing the divergence.** The section on the optimal discriminator showed that the
Jensen-Shannon divergence is flat
between distributions with disjoint supports, which is the situation of any image GAN at the
start of training. Wasserstein GANs
[[Arjovsky, Chintala & Bottou, 2017]](https://arxiv.org/abs/1701.07875) replace it with the
earth-mover distance, which still knows how far apart two disjoint distributions are, and with
it the discriminator, now called a critic, produces a loss that actually decreases as the
samples improve.

**Measuring what the losses cannot.** The Fréchet Inception Distance
[[Heusel et al., NeurIPS 2017]](https://arxiv.org/abs/1706.08500) compares the two
distributions through the first two moments of their features in a fixed pretrained network.
It is external to the game, so it compares across runs and across models, and it moves when a
mode is dropped.

**Going the other way.** The VAE lab builds a generative model that is a probability
distribution rather than a sampler: it has a density, an encoder, and a loss worth reading. It
opens on the closing figure of Part II.
